In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
search_tool = {
  "type": "function",
  "name": "search",
  "description": "Search the FAQ database for the answer to the user's question",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Search query text to look up in the course FAQ"
      }
    },
    "required": ["query"],
    "additionalProperties": False
  }
}


In [8]:
import os
from rag_helper import RAGHelper
from ingest import load_faq_data
from sqlitesearch import TextSearchIndex

documents = load_faq_data()

index = TextSearchIndex(
  text_fields=['question', 'section', 'answer'],
  keyword_fields=['course'],
  db_path='faq.db'
)

if not os.path.exists('faq.db') or os.path.getsize('faq.db') == 0:
  index.fit(documents)
  index.save() 

assistant = RAGHelper(index, openai_client, tools=[search_tool])

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


In [9]:
# answer = assistant.rag('can I still take this course?')
#answer = assistant.agentic_rag('can I still take this course?')
# answer = assistant.rag('how do i run olama locally?')
# answer = assistant.agentic_rag('how do i run olama locally?')


In [ ]:
# print(answer)

NameError: name 'answer' is not defined

In [ ]:
#bad_answer = assistant.agentic_rag('what is the weather in tokyo?')

Iteration 1
ASSISTANT:
I don't know


In [10]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [11]:
agent_tools = Tools()
# if you define the tool with docstring and types, then add_tool can infer
# craft the tool from the docstring
agent_tools.add_tool(RAGHelper.search, search_tool)

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
  tools=agent_tools,
  developer_prompt=assistant.instructions,
  chat_interface=chat_interface,
  llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [19]:
result = runner.loop(
  prompt='what is the weather in tokyo?',
  callback=callback
)

result.cost
result.all_messages

-> Response received


[EasyInputMessage(content='\nYou are a helpful assistant that can answer questions about the course given the provided context.\nUse the context to find relevant information and provide accurate answers.\nIf an answer is not found in the context, respond with "I don\'t know"\n', role='developer', phase=None, type=None),
 EasyInputMessage(content='what is the weather in tokyo?', role='user', phase=None, type=None),
 ResponseOutputMessage(id='msg_073c3de6810a46bb006a5fc9bec9308196ab3cd7e634373d8d', content=[ResponseOutputText(annotations=[], text='I don’t know', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [20]:
result2 = runner.loop(
  prompt='how do i run a different model',
  previous_messages=result.all_messages,
  callback=callback
)

-> Response received


-> Response received
